# Stale Jobs - Scalable Deletion

Deletes jobs by **Job ID** via the Databricks **Jobs REST API 2.1** (`/api/2.1/jobs/get`, `/api/2.1/jobs/create`, `/api/2.1/jobs/delete`). Built for ~1000 jobs.

**Safety:**
* `DRY_RUN` defaults True
* `confirm_workspace` required
* UC volume widget (defaults to an existing test volume)
* JSONL backup on that volume before delete
* Bad IDs skipped
* Thread pool + 9 req/s cap + 429/5xx backoff

Attach a **UC-enabled** cluster (or serverless). Do not use a DBFS-only cluster.

**Customer:** widgets → DRY_RUN → config → load IDs → validate/backup → delete. Skip test-job create.

Disclaimer: guidance only. Validate against your environment and change-management process. Databricks is not responsible for data loss or job disruption.

## Confirm this workspace
Set the **confirm_workspace** widget (top of the notebook) to **this** workspace. Any of these formats work — spaces around dots are ignored:

| Enter this | Example |
|---|---|
| Hostname prefix | `your-workspace` |
| Full hostname | `your-workspace.cloud.databricks.com` |
| URL | `https://your-workspace.cloud.databricks.com` |

Leave it blank and the next cells will stop. A value for a *different* workspace also stops.

In [0]:
dbutils.widgets.text(
    "confirm_workspace",
    "",
    "Workspace: prefix, hostname, or https:// URL (e.g. your-workspace)",
)

# Opt-in guard: test-job creation is OFF by default so a batch of 25 stale
# jobs is never created by accident. Set to "yes" only in a test workspace.
dbutils.widgets.dropdown(
    "create_test_jobs",
    "no",
    ["no", "yes"],
    "Create 25 disposable TEST jobs? (test workspace only)",
)

In [0]:
DRY_RUN = True  # Master safety switch. Set False (and re-run THIS cell) only after a dry-run looks right.


## Backup volume
Job definitions are written to a **Unity Catalog Volume** (not DBFS). Set **uc_volume** at the top. Either format works:

| Enter this | Example |
|---|---|
| Three-level name | `workspace.my_schema.my_volume` |
| Filesystem path | `/Volumes/workspace/my_schema/my_volume` |
| Path + folder | `/Volumes/workspace/my_schema/my_volume/job_deletion_backup` |

Default is your configured volume. Backups land in a `job_deletion_backup/` folder inside the volume.

In [0]:
dbutils.widgets.text(
    "uc_volume",
    "workspace.my_schema.my_volume",
    "UC volume: catalog.schema.volume  OR  /Volumes/catalog/schema/volume",
)

BACKUP_SUBDIR = "job_deletion_backup"

def resolve_uc_volume_path(raw):
    """Accept catalog.schema.volume or /Volumes/catalog/schema/volume[/folder]."""
    s = (raw or "").strip().replace("dbfs:", "").rstrip("/")
    if not s:
        raise RuntimeError(
            "Set the uc_volume widget. Examples:\n"
            "  workspace.my_schema.my_volume\n"
            "  /Volumes/workspace/my_schema/my_volume"
        )
    if s.startswith("/Volumes/"):
        parts = s.strip("/").split("/")
        if len(parts) < 4:
            raise RuntimeError(
                "Volume path must be /Volumes/<catalog>/<schema>/<volume>[/folder]. "
                f"Got {raw!r}"
            )
        volume_root = "/" + "/".join(parts[:4])
        rest = "/".join(parts[4:])
        backup_path = s if rest else f"{volume_root}/{BACKUP_SUBDIR}"
        return volume_root, backup_path
    bits = [b.strip() for b in s.split(".") if b.strip()]
    if len(bits) != 3:
        raise RuntimeError(
            "Use catalog.schema.volume or /Volumes/catalog/schema/volume.\n"
            "  workspace.my_schema.my_volume\n"
            "  /Volumes/workspace/my_schema/my_volume"
        )
    catalog, schema, volume = bits
    volume_root = f"/Volumes/{catalog}/{schema}/{volume}"
    return volume_root, f"{volume_root}/{BACKUP_SUBDIR}"

VOLUME_ROOT, BACKUP_PATH = resolve_uc_volume_path(dbutils.widgets.get("uc_volume"))
print("Volume root:", VOLUME_ROOT)
print("Backup folder:", BACKUP_PATH)
print("(Three-level name and /Volumes/... path are both OK.)")

In [0]:
MAX_WORKERS      = 8
RATE_LIMIT       = 9          # Jobs API limit is 10/s; leave headroom
REQUEST_TIMEOUT  = 30

import json, time, os, datetime, threading
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def normalize_ws_host(raw):
    """Accept prefix, hostname, or URL. Returns lowercase hostname (no scheme/path)."""
    s = "".join((raw or "").strip().lower().split())
    if s.startswith("https://"):
        s = s[8:]
    elif s.startswith("http://"):
        s = s[7:]
    return s.split("/")[0].split("?")[0].split("#")[0].rstrip(".")

def ws_keys(host):
    h = normalize_ws_host(host)
    keys = {h, h.split(".")[0]}
    if "." not in h:
        keys.add(h + ".cloud.databricks.com")
    return keys

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
actual_host = normalize_ws_host(ctx.browserHostName().get())
confirm = normalize_ws_host(dbutils.widgets.get("confirm_workspace"))
if not confirm:
    raise RuntimeError(
        "Set the confirm_workspace widget first. Examples that all work:\n"
        "  your-workspace\n"
        "  your-workspace.cloud.databricks.com\n"
        "  https://your-workspace.cloud.databricks.com"
    )
if ws_keys(confirm).isdisjoint(ws_keys(actual_host)):
    raise RuntimeError(
        f"confirm_workspace {confirm!r} does not match this workspace ({actual_host}). "
        "Use the prefix, hostname, or URL of the workspace this notebook is attached to."
    )
print(f"Workspace confirmed: {actual_host}  (entered {confirm})")

if "BACKUP_PATH" not in dir() or not BACKUP_PATH:
    raise RuntimeError("Run the UC volume cell first (uc_volume widget).")
if BACKUP_PATH.startswith("/dbfs"):
    raise RuntimeError("BACKUP_PATH must be a Unity Catalog Volume path (/Volumes/...), not DBFS.")

HOST = "https://" + ctx.browserHostName().get()
TOKEN = ctx.apiToken().get()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
print(f"Workspace host: {HOST}")
print(f"DRY_RUN={DRY_RUN}  backup={BACKUP_PATH}")

class RateLimiter:
    """Cap calls at `rate` per second across all threads (thread-safe)."""
    def __init__(self, rate):
        self.min_interval = 1.0 / rate
        self.lock = threading.Lock()
        self.next_time = 0.0

    def acquire(self):
        with self.lock:
            now = time.monotonic()
            wait = self.next_time - now
            if wait > 0:
                time.sleep(wait)
            self.next_time = max(now, self.next_time) + self.min_interval

limiter = RateLimiter(RATE_LIMIT)

def api_call(method, path, payload=None, max_retries=6):
    """Jobs/Clusters REST helper: rate-limited + 429/5xx backoff (Retry-After)."""
    url = f"{HOST}{path}"
    resp = None
    for attempt in range(max_retries):
        limiter.acquire()
        resp = requests.request(
            method, url, headers=HEADERS,
            data=json.dumps(payload) if payload is not None else None,
            timeout=REQUEST_TIMEOUT,
        )
        if resp.status_code == 429 or resp.status_code >= 500:
            wait = float(resp.headers.get("Retry-After", min(2 ** attempt, 30)))
            time.sleep(wait)
            continue
        return resp
    return resp

In [0]:
# INTERNAL TEST ONLY. Guarded: nothing runs unless create_test_jobs == "yes".
N_TEST_JOBS = 25
CREATE_TEST_JOBS = dbutils.widgets.get("create_test_jobs").strip().lower() == "yes"

if not CREATE_TEST_JOBS:
    print(
        f"Skipped test-job creation. create_test_jobs='no'. "
        f"No jobs created. Set the create_test_jobs widget to 'yes' to create "
        f"{N_TEST_JOBS} disposable jobs (test workspace only)."
    )
else:
    created_ids = []
    for i in range(N_TEST_JOBS):
        # Serverless compute: no new_cluster or existing_cluster_id needed
        payload = {
            "name": f"ZZZ-stale-job-test-{i}-{datetime.datetime.utcnow():%Y%m%d%H%M%S}",
            "tasks": [{
                "task_key": "noop",
                "notebook_task": {"notebook_path": ctx.notebookPath().get()},
                # No compute specification = defaults to serverless
            }],
        }
        r = api_call("POST", "/api/2.1/jobs/create", payload)
        if not r.ok:
            raise RuntimeError(f"Job create failed ({r.status_code}): {r.text}")
        created_ids.append(r.json()["job_id"])

    print(f"Created {len(created_ids)} test jobs.")
    job_ids_to_delete = created_ids

### Customer: load Job IDs
Skip the test-create cell. Uncomment and set IDs, or read a CSV from the UC Volume.

In [0]:
# ── Cell 1′ (CUSTOMER): load the real list of Job IDs ───────────────────────
# job_ids_to_delete = [
#     123456789012345,
#     234567890123456,
# ]
#
# import pandas as pd
# df = pd.read_csv(f"{VOLUME_ROOT}/stale_jobs.csv")
# job_ids_to_delete = df["job_id"].astype(int).tolist()
#
# job_ids_to_delete = [int(j) for j in job_ids_to_delete]
# assert len(set(job_ids_to_delete)) == len(job_ids_to_delete), "Duplicate IDs in list"
# print(f"Loaded {len(job_ids_to_delete)} job IDs to process.")

In [0]:
def volume_reachable(root):
    try:
        dbutils.fs.ls(root)
        return True
    except Exception:
        return False

if not volume_reachable(VOLUME_ROOT):
    raise RuntimeError(
        f"Volume not reachable: {VOLUME_ROOT}\n"
        "Fix the uc_volume widget (catalog.schema.volume or /Volumes/catalog/schema/volume) "
        "and confirm the cluster has USE CATALOG / USE SCHEMA / READ VOLUME / WRITE VOLUME.\n"
        f"Example: workspace.my_schema.my_volume"
    )

# Only mkdir folders *inside* the volume. Never mkdir the volume mount itself (Errno 95).
if BACKUP_PATH.rstrip("/") != VOLUME_ROOT:
    dbutils.fs.mkdirs(BACKUP_PATH)

stamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
backup_name = f"jobs_backup_{stamp}.jsonl"
backup_file = f"{BACKUP_PATH.rstrip('/')}/{backup_name}"

valid, not_found = [], []
backup_lines = []

def fetch(job_id):
    r = api_call("GET", f"/api/2.1/jobs/get?job_id={job_id}")
    return job_id, (r.json() if r.status_code == 200 else None), r.status_code

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    for job_id, body, code in ex.map(fetch, job_ids_to_delete):
        if body:
            valid.append(job_id)
            backup_lines.append(json.dumps(body) + "\n")
        else:
            not_found.append((job_id, code))

# Write directly to volume using dbutils
dbutils.fs.put(backup_file, "".join(backup_lines), overwrite=True)
print(f"Validated: {len(valid)} found | {len(not_found)} not found/inaccessible")
print(f"Backup written to: {backup_file}")
if not_found:
    print("NOT FOUND (will be skipped):", not_found[:20], "..." if len(not_found) > 20 else "")

In [0]:
def delete_one(job_id):
    if DRY_RUN:
        return job_id, "dry-run", None
    r = api_call("POST", "/api/2.1/jobs/delete", {"job_id": job_id})
    return job_id, ("ok" if r.status_code == 200 else "error"), \
           (None if r.status_code == 200 else f"{r.status_code}: {r.text[:200]}")

results = {"ok": [], "error": [], "dry-run": []}
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(delete_one, j) for j in valid]
    for n, fut in enumerate(as_completed(futures), 1):
        job_id, status, err = fut.result()
        results[status].append(job_id if status != "error" else (job_id, err))
        if n % 50 == 0:
            print(f"  processed {n}/{len(valid)}")

print(f"\n{'DRY-RUN — nothing deleted' if DRY_RUN else 'DELETE COMPLETE'}")
print(f"  would-delete/deleted: {len(results['dry-run']) + len(results['ok'])}")
print(f"  errors: {len(results['error'])}")
for jid, err in results["error"][:20]:
    print(f"    job {jid}: {err}")

In [0]:
# After a live run (DRY_RUN=False), confirm jobs are gone.
still_present = [j for j in valid if api_call("GET", f"/api/2.1/jobs/get?job_id={j}").status_code == 200]
print(f"Still present after delete: {len(still_present)}")
print(still_present[:20])